In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
csv_path = os.path.join(path, 'Q1_data.csv')
data = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
data.head()

In [ ]:
# Task 3: Write your code here:
data.info()

In [ ]:
# Task 4: Write your code here:
data.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=20, edgecolor='yellow')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(data, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = data.drop('Order_ID', axis=1)
df.head()

In [ ]:
# Task 2: Write your code here:
df.isnull().sum(0)

from sklearn.preprocessing import LabelEncoder

col_with_missing_values = ["Weather" , "Traffic_Level" , "Time_of_Day" , "Courier_Experience_yrs"]

for col in col_with_missing_values:
    df[col] = df[col].fillna('unknown')


df = df.dropna(subset=["Delivery_Time"])

df.head()
df.isnull().sum()

In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):

  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")

  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")

  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.

In [ ]:
df.info()

In [ ]:
categorical_cols = df.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[[col]])

df.head()

In [ ]:
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].replace("unknown" , 0.0)

In [ ]:
df['Courier_Experience_yrs']

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df.drop('Delivery_Time', axis=1)
y = df["Delivery_Time"]

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
# Task 2,3,4,5: Write your code here:
model = RandomForestRegressor(max_depth=5)
n_plits = 4
skf = StratifiedKFold(n_splits=n_plits, shuffle=True, random_state=42)

In [ ]:
mse = []

In [ ]:
df.info()

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

  print(f"\nFold {fold_idx + 1}/{n_plits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models

  print(f"Training ...")
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  mse.append(mean_squared_error(y_test, y_pred))

In [ ]:
# Gather importances from the models (from the last fold)
importances = {
    "RandomForest" : model.feature_importances_
}

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
df['Delivery_Time'].hist()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings('ignore')

rf_model = RandomForestRegressor(max_depth=5, random_state=42)
cat_model = CatBoostRegressor(iterations=100, learning_rate=0.1, depth=5, random_seed=42, verbose=0)

mae_scores = []

n_splits = 4
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

categorical_features_indices = np.where(X.dtypes == 'object')[0]

for fold_idx, (train_index, test_index) in enumerate(kf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    # Split data
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    print("Training RandomForest...")
    rf_model.fit(X_train, y_train)
    rf_preds = rf_model.predict(X_test)

    print("Training CatBoost...")
    cat_features = X_train.select_dtypes(include=['object']).columns.tolist()
    if cat_features:
        cat_model.fit(X_train, y_train, cat_features=cat_features, verbose=0)
    else:
        cat_model.fit(X_train, y_train, verbose=0)
    cat_preds = cat_model.predict(X_test)

    # Average predictions
    avg_preds = (rf_preds + cat_preds) / 2

    # Calculate MAE
    mae = mean_absolute_error(y_test, avg_preds)
    mae_scores.append(mae)
    print(f"MAE for Fold {fold_idx + 1}: {mae:.2f}")

print(f"\nAverage MAE across all folds: {np.mean(mae_scores):.2f}")